# Déduplication des acteurs de l'économie circulaire

Reproduction de l'algorithme de [data-inclusion](https://github.com/gip-inclusion/data-inclusion/tree/main/deduplication)
appliqué aux acteurs VueActeur de notre projet.

**Approche** : utilisation de la bibliothèque [dedupe](https://github.com/dedupeio/dedupe) avec :
- **Exemples positifs** : clusters parent/enfants déjà validés
- **Exemples négatifs** : suggestions de clustering rejetées
- **Features** : nom, adresse, code_postal, ville, siret, siren, telephone, email, localisation géographique

## Note pour plus tard :)

- essayer avec 1000 exemples positifs et 500 négatifs
- tester sur 1000 autres positifs et 500 négatifs
- vérifier le taux de réussite


In [ ]:
! export LDFLAGS="-L$(brew --prefix gettext)/lib"
! export CPPFLAGS="-I$(brew --prefix gettext)/include"

! uv sync


## 1. Initialiser l'environnement

In [ ]:
import sys
import os
from pathlib import Path

# Notebook : data-platform/notebooks
notebooks_dir = Path.cwd()
project_root = notebooks_dir.parent        # => data-platform

sys.path.append(str(project_root))
sys.path.append(str(project_root / "dags"))

from dags.utils.django import django_setup_full

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "True"

django_setup_full()

In [ ]:
import dedupe
import pandas as pd
import numpy as np
from unidecode import unidecode
from collections import defaultdict

## 2. Charger les données depuis VueActeur

In [ ]:
from django.db.models import F
from qfdmo.models import VueActeur

# Deux jeux de données :
# - all_acteurs : tous les acteurs (y compris dédupliqués) pour l'entraînement
# - visible_acteurs : uniquement les acteurs visibles pour l'inférence
all_acteurs_qs = (
    VueActeur.objects.all()
    .select_related("source", "acteur_type")
    .annotate(acteur_type__libelle=F("acteur_type__libelle"))
)
visible_acteurs_qs = (
    VueActeur.objects.get_visible_acteurs()
    .select_related("source", "acteur_type")
    .annotate(acteur_type__libelle=F("acteur_type__libelle"))
)

print(f"Tous les acteurs (entraînement) : {all_acteurs_qs.count()}")
print(f"Acteurs visibles (inférence)    : {visible_acteurs_qs.count()}")
print(
    f"Acteurs dédupliqués (non visibles) : {all_acteurs_qs.count() - visible_acteurs_qs.count()}"
)

In [ ]:
# DataFrame complet (tous les acteurs) pour l'entraînement
acteurs_values = all_acteurs_qs.values(
    "identifiant_unique",
    "uuid",
    "nom",
    "adresse",
    "code_postal",
    "ville",
    "siret",
    "siren",
    "telephone",
    "email",
    "latitude",
    "longitude",
    "source__libelle",
    "acteur_type__libelle",
    "nom_commercial",
    "nom_officiel",
)

df = pd.DataFrame(list(acteurs_values))
df = df.set_index("identifiant_unique")
print(f"DataFrame complet : {df.shape[0]} acteurs, {df.shape[1]} colonnes")

# Index des acteurs visibles (pour l'inférence)
visible_ids = set(
    visible_acteurs_qs.values_list("identifiant_unique", flat=True)
)
print(f"Acteurs visibles : {len(visible_ids)}")
df.head()

## 3. Prétraitement des données

Normalisation inspirée de data-inclusion : lowercase, suppression des accents, nettoyage des espaces.
Les coordonnées GPS sont converties au format `(latitude, longitude)` attendu par `dedupe.variables.LatLong`.

In [ ]:
def normalize_string(value):
    """Normalise une chaîne : lowercase, suppression accents, strip."""
    if pd.isna(value) or not value:
        return None
    value = str(value).strip().lower()
    value = unidecode(value)
    # Supprimer les espaces multiples
    value = " ".join(value.split())
    return value if value else None


def normalize_phone(value):
    """Normalise un numéro de téléphone français."""
    if pd.isna(value) or not value:
        return None
    phone = str(value).strip()
    # Garder uniquement les chiffres et le +
    phone = "".join(c for c in phone if c.isdigit() or c == "+")
    # Format français : 0X XX XX XX XX
    if phone.startswith("33"):
        phone = "0" + phone[2:]
    elif phone.startswith("+33"):
        phone = "0" + phone[3:]
    return phone if len(phone) >= 10 else None


def normalize_siret(value):
    """Normalise un SIRET (14 chiffres)."""
    if pd.isna(value) or not value:
        return None
    siret = "".join(c for c in str(value) if c.isdigit())
    return siret if len(siret) == 14 else None


def extract_siren(siret):
    """Extrait le SIREN (9 premiers chiffres du SIRET)."""
    if pd.isna(siret) or not siret:
        return None
    digits = "".join(c for c in str(siret) if c.isdigit())
    return digits[:9] if len(digits) >= 9 else None


# Appliquer les normalisations
for col in ["nom", "adresse", "ville", "nom_commercial", "nom_officiel"]:
    df[col] = df[col].apply(normalize_string)

df["email"] = df["email"].apply(lambda x: normalize_string(x) if pd.notna(x) else None)
df["telephone"] = df["telephone"].apply(normalize_phone)
df["siret"] = df["siret"].apply(normalize_siret)
df["siren"] = df.apply(
    lambda row: extract_siren(row["siret"]) if row["siret"] else extract_siren(row["siren"]),
    axis=1,
)

# Construire la colonne location (tuple lat, lon) pour dedupe LatLong
# Modulo 90/180 pour corriger les coordonnées hors bornes
def normalize_location(row):
    lat, lon = row["latitude"], row["longitude"]
    if pd.isna(lat) or pd.isna(lon) or (lat == 0 and lon == 0):
        return None
    lat = float(lat) % 90 if float(lat) > 90 else float(lat) % -90 if float(lat) < -90 else float(lat)
    lon = float(lon) % 180 if float(lon) > 180 else float(lon) % -180 if float(lon) < -180 else float(lon)
    return (lat, lon)

df["location"] = df.apply(normalize_location, axis=1)

print("Taux de remplissage après normalisation :")
for col in ["nom", "adresse", "code_postal", "ville", "siret", "siren",
            "telephone", "email", "location"]:
    pct = df[col].notna().mean() * 100
    print(f"  {col:15s}: {pct:5.1f}%")

## 4. Construire les paires labélisées

### 4.1 Exemples positifs : clusters existants (parent/enfants)

Chaque parent avec ses enfants forme un cluster validé. Toutes les paires au sein d'un cluster sont des **match**.

In [ ]:
from itertools import combinations

# Récupérer les parents visibles avec leurs enfants en une seule requête
parents = (
    VueActeur.objects.get_visible_acteurs()
    .filter(est_parent=True)
    .prefetch_related("duplicats")
)
print(f"Nombre de parents : {parents.count()}")

# Construire les paires positives à partir des clusters
match_pairs = []
skipped = 0

for parent in parents:
    # We don't want to include the parent in the cluster combinations
    cluster_ids = [ #parent.identifiant_unique] + [
        e.identifiant_unique for e in parent.duplicats.all()
    ]

    # FIXME : non car les enfants ne font pas parti de la DF
    # FIXME : il faut peut-être ajouter les enfants dans la dataframe ?
    #  Ne garder que les IDs présents dans notre DataFrame
    #cluster_ids_in_df = [identifiant_unique for identifiant_unique in cluster_ids if identifiant_unique in df.index]

    if len(cluster_ids) >= 2:
        for id_a, id_b in combinations(cluster_ids, 2):
            match_pairs.append((id_a, id_b))
    else:
        skipped += 1

print(f"Paires positives (match) : {len(match_pairs)}")
print(f"Clusters ignorés (< 2 acteurs dans le DF) : {skipped}")

### 4.2 Exemples négatifs : suggestions de clustering rejetées

Les suggestions de déduplication rejetées par les validateurs contiennent des paires que l'humain a jugé comme **non-doublons**.

In [ ]:
from data.models.suggestion import Suggestion

# Récupérer les suggestions de clustering rejetées
rejected_suggestions = Suggestion.objects.filter(
    statut="REJETEE",
    suggestion_cohorte__type_action="CLUSTERING",
).select_related("suggestion_cohorte")

print(f"Suggestions de clustering rejetées : {rejected_suggestions.count()}")

# Extraire les paires distinctes à partir des suggestions rejetées
distinct_pairs = []
rejected_details = []

for rejected in rejected_suggestions:
    suggestion_data = rejected.suggestion
    contexte_data = rejected.contexte

    # Extraire les identifiants depuis le contexte (liste de dicts avec identifiant_unique)
    ids_in_suggestion = []

    # Le contexte contient les acteurs impliqués dans le cluster proposé
    if isinstance(contexte_data, dict) and "fuzzy_details" in contexte_data:
        ids_in_suggestion = [
            item["identifiant_unique"]
            for item in contexte_data["fuzzy_details"]
            if isinstance(item, dict) and "identifiant_unique" in item
        ]
    elif isinstance(contexte_data, list):
        ids_in_suggestion = [
            item["identifiant_unique"]
            for item in contexte_data
            if isinstance(item, dict) and "identifiant_unique" in item
        ]

    # Ne garder que les IDs présents dans le DataFrame
    ids_in_df = [identifiant_unique for identifiant_unique in ids_in_suggestion if identifiant_unique in df.index]

    if len(ids_in_df) >= 2:
        for id_a, id_b in combinations(ids_in_df, 2):
            distinct_pairs.append((id_a, id_b))
        rejected_details.append({
            "suggestion_id": rejected.id,
            "nb_acteurs": len(ids_in_df),
            "ids": ids_in_df,
        })

# Dédoublonner les paires
distinct_pairs = list(set(distinct_pairs))

print(f"Paires négatives (distinct) : {len(distinct_pairs)}")
print(f"Suggestions exploitables : {len(rejected_details)}")

# Afficher quelques exemples
if rejected_details:
    print(f"\nExemple de suggestion rejetée :")
    for i in range(10):
        print(f"  IDs : {rejected_details[i]['ids'][:5]}")


### 4.3 Résumé des labels et formatage pour dedupe

In [ ]:
# Vérifier qu'il n'y a pas de paires contradictoires (à la fois match et distinct)
match_set = set(tuple(sorted(p)) for p in match_pairs)
distinct_set = set(tuple(sorted(p)) for p in distinct_pairs)
contradictions = match_set & distinct_set

print(f"Paires match (dédoublonnées) : {len(match_set)}")
print(f"Paires distinct (dédoublonnées) : {len(distinct_set)}")
print(f"Contradictions : {len(contradictions)}")

# Retirer les contradictions des deux ensembles (traitement "unsure" comme data-inclusion)
if contradictions:
    print("Les contradictions seront retirées des deux ensembles.")
    match_set -= contradictions
    distinct_set -= contradictions

print(f"\nPaires finales :")
print(f"  match   : {len(match_set)}")
print(f"  distinct: {len(distinct_set)}")
print(f"  total   : {len(match_set) + len(distinct_set)}")

## 5. Préparer le dictionnaire de données pour dedupe

Le format attendu par dedupe est un `dict[str, dict]` où chaque clé est l'ID unique et la valeur est un dict des champs.

In [ ]:
# Colonnes utilisées pour la déduplication
DEDUPE_FIELDS = ["identifiant_unique","uuid","nom", "adresse", "adresse_complement",
                 "code_postal", "ville", "siret", "siren",
                 "telephone", "email", "location",
                 "acteur_type__libelle"]


def row_to_dedupe_record(row):
    """Convertit une ligne du DataFrame en dict pour dedupe.
    Les valeurs None/NaN/chaînes vides sont converties en None
    (dedupe gère has_missing)."""
    record = {}
    for field in DEDUPE_FIELDS:
        val = row.get(field)
        if isinstance(val, tuple):
            record[field] = val if val is not None else None
        elif pd.isna(val) or val == "":
            record[field] = None
        else:
            record[field] = val
    return record


# Dict complet (tous les acteurs) pour l'entraînement
data_dict_all = {}
for idx, row in df.iterrows():
    data_dict_all[idx] = row_to_dedupe_record(row)

# Dict restreint aux acteurs visibles pour l'inférence
data_dict_visible = {k: v for k, v in data_dict_all.items() if k in visible_ids}

print(f"Enregistrements pour entraînement : {len(data_dict_all)}")
print(f"Enregistrements pour inférence    : {len(data_dict_visible)}")

# Aperçu d'un enregistrement
sample_key = list(data_dict_all.keys())[0]
print(f"\nExemple ({sample_key}) :")
for k, v in data_dict_all[sample_key].items():
    print(f"  {k}: {v}")

## 6. Définir le modèle dedupe

Définition des variables inspirée de data-inclusion, adaptée aux champs VueActeur :
- `nom` : String avec CRF (Conditional Random Field) pour capturer la structure interne des noms
- `adresse`, `ville` : String avec gestion des valeurs manquantes
- `location` : LatLong (distance haversine)
- `siret`, `siren`, `telephone`, `email` : ShortString
- `code_postal` : Exact
- Interactions : `siret * siren` et `ville * adresse * location * code_postal`

In [ ]:
# Définition des variables dedupe (inspirée de data-inclusion)
# Note: dedupe 3.x Interaction requiert le format complet "(field: Type)"
# FIXME: ajouté acteur_type
fields = [
    dedupe.variables.String("nom", crf=True),
    dedupe.variables.String("adresse", has_missing=True),
    dedupe.variables.String("adresse_complement", has_missing=True),
    dedupe.variables.String("email", has_missing=True),
    dedupe.variables.LatLong("location", has_missing=True),
    dedupe.variables.Exact("code_postal"),
    dedupe.variables.ShortString("ville", has_missing=True),
    dedupe.variables.ShortString("acteur_type__libelle"),
    dedupe.variables.ShortString("siren", has_missing=True),
    dedupe.variables.ShortString("siret", has_missing=True),
    dedupe.variables.ShortString("telephone", has_missing=True),
    # Interactions
    dedupe.variables.Interaction(
        "(siret: ShortString)", "(siren: ShortString)"
    ),
    dedupe.variables.Interaction(
        "(adresse: String)", "(adresse_complement: String)",
        "(code_postal: Exact)", "(ville: ShortString)",
    ),
]

# Créer le déduplicateur
# num_cores=1 pour éviter les erreurs de sérialisation joblib dans Jupyter
deduper = dedupe.Dedupe(fields, num_cores=1)
print("Modèle dedupe créé avec les variables suivantes :")
for f in fields:
    print(f"  - {f}")

In [ ]:
# Sous-échantillonner les données pour prepare_training (performance)
# On prend tous les acteurs impliqués dans les labels + un échantillon aléatoire
labeled_ids = set()
for a, b in match_set | distinct_set:
    labeled_ids.add(a)
    labeled_ids.add(b)

print(f"Acteurs impliqués dans les labels : {len(labeled_ids)}")

# Sous-échantillon : tous les labélisés + 1 acteur sur 10 pour le reste
import hashlib

def stable_hash(s):
    return int(hashlib.md5(str(s).encode()).hexdigest(), 16)

sample_ids = set()
for idx in df.index:
    if idx in labeled_ids or stable_hash(idx) % 1000 == 0:
        sample_ids.add(idx)

data_sample = {k: v for k, v in data_dict_all.items() if k in sample_ids}
print(f"Échantillon pour l'entraînement : {len(data_sample)} acteurs "
      f"({len(data_sample)/len(data_dict_all)*100:.1f}%)")

In [ ]:
import random

# Construire les paires labélisées au format dedupe
# et les injecter AVANT prepare_training via deduper.training_pairs


def pairs_to_dedupe_format(pair_set, data, max_pairs=None):
    """Convertit un set de paires (id_a, id_b) en format dedupe."""
    pairs = list(pair_set)
    if max_pairs and len(pairs) > max_pairs:
        pairs = random.sample(pairs, max_pairs)
    result = []
    for id_a, id_b in pairs:
        if id_a in data and id_b in data:
            result.append((data[id_a], data[id_b]))
    return result


# Équilibrer les paires : limiter match à 3x le nombre de distinct
distinct_pairs = pairs_to_dedupe_format(distinct_set, data_dict_all)

# FIXME : a prendre en random
max_match = max(len(distinct_pairs) * 3, 1000)
match_pairs = pairs_to_dedupe_format(match_set, data_dict_all, max_match)

# Injecter les labels AVANT prepare_training
# (prepare_training les utilisera pour construire les index de blocking)
deduper.training_pairs = {
    "match": match_pairs,
    "distinct": distinct_pairs,
}

print(f"Labels injectés dans deduper.training_pairs :")
print(f"  match   : {len(match_pairs)}")
print(f"  distinct: {len(distinct_pairs)}")


In [ ]:
# Préparer l'entraînement
# prepare_training utilise deduper.training_pairs pour indexer
# les paires labélisées dans l'active learner
deduper.prepare_training(data_sample)
print("prepare_training terminé")


In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Entraîner le modèle (régression logistique régularisée)
deduper.train()
print("Entraînement terminé !")

# Sauvegarder les labels d'entraînement
TRAINING_PATH = 'deduplication_training.json'
with open(TRAINING_PATH, 'w') as f:
    deduper.write_training(f)
print(f"Labels sauvegardés dans {TRAINING_PATH}")


## 8. Recherche du seuil optimal

Comme data-inclusion, on fait une recherche en grille sur les paramètres `recall` et `threshold`
pour trouver le meilleur compromis précision/rappel.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Construire un dataset de paires labélisées pour évaluation
all_labeled = []
for id_a, id_b in match_set:
    if id_a in data_dict_all and id_b in data_dict_all:
        all_labeled.append((id_a, id_b, 1))  # match
for id_a, id_b in distinct_set:
    if id_a in data_dict_all and id_b in data_dict_all:
        all_labeled.append((id_a, id_b, 0))  # distinct

print(f"Total paires labélisées : {len(all_labeled)}")

# Split train/test stratifié
labels_df_eval = pd.DataFrame(all_labeled, columns=["id_a", "id_b", "label"])
train_pairs, test_pairs = train_test_split(
    labels_df_eval, test_size=0.2, stratify=labels_df_eval["label"], random_state=42
)

print(f"Train : {len(train_pairs)} (match: {train_pairs['label'].sum()}, "
      f"distinct: {(train_pairs['label']==0).sum()})")
print(f"Test  : {len(test_pairs)} (match: {test_pairs['label'].sum()}, "
      f"distinct: {(test_pairs['label']==0).sum()})")

In [ ]:
# Scorer les paires de test avec le modèle entraîné
# dedupe.score() attend des paires au format ((id_a, record_a), (id_b, record_b))
# et retourne un structured array avec champs 'pairs' et 'score'
test_record_pairs = []
test_id_pairs = []
for _, row in test_pairs.iterrows():
    id_a, id_b = row['id_a'], row['id_b']
    test_record_pairs.append(
        ((id_a, data_dict_all[id_a]), (id_b, data_dict_all[id_b]))
    )
    test_id_pairs.append((id_a, id_b))

scored = deduper.score(test_record_pairs)

# Construire un dict (id_a, id_b) -> score pour lookup
score_lookup = {}
for row in scored:
    pair = tuple(row['pairs'])
    score_lookup[pair] = float(row['score'])

# Aligner les scores avec les paires de test
scores_aligned = []
for id_a, id_b in test_id_pairs:
    # Le score peut être sous (id_a, id_b) ou (id_b, id_a)
    s = score_lookup.get((id_a, id_b), score_lookup.get((id_b, id_a), 0.0))
    scores_aligned.append(s)
scores_arr = np.array(scores_aligned)
true_labels = test_pairs['label'].values

print(f'Paires scorées par dedupe : {len(scored)} / {len(test_record_pairs)}')
print(f'Scores > 0 : {(scores_arr > 0).sum()}')

# Tester différents seuils
print("\nSeuil  | Précision | Rappel | F1-score | True pos. | Faux pos. | Faux nég.")
print("-------|-----------|--------|----------|-----------|-----------|-----------")
best_f1 = 0
best_threshold = 0.5

for threshold in np.arange(0.1, 1.0, 0.05):
    predicted = (scores_arr >= threshold).astype(int)
    tp = ((predicted == 1) & (true_labels == 1)).sum()
    fp = ((predicted == 1) & (true_labels == 0)).sum()
    fn = ((predicted == 0) & (true_labels == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

    print(threshold)
    print(f" {threshold:.2f}  |   {precision:.3f}   | {recall:.3f}  |   {f1:.3f}  |   {tp:6d}  |   {fp:6d}  |  {fn:6d}")

print(f"\nMeilleur seuil : {best_threshold:.2f} (F1={best_f1:.3f})")


In [ ]:
best_threshold = 0.95

In [ ]:
# Rapport de classification détaillé au meilleur seuil
predicted = (scores_arr >= best_threshold).astype(int)
print(f"=== Rapport au seuil {best_threshold:.2f} ===\n")
print(classification_report(
    true_labels, predicted,
    target_names=["distinct", "match"],
    digits=3,
))
print("Matrice de confusion :")
cm = confusion_matrix(true_labels, predicted)
print(f"  TN={cm[0][0]:4d}  FP={cm[0][1]:4d}")
print(f"  FN={cm[1][0]:4d}  TP={cm[1][1]:4d}")

## 9. Déduplication sur l'ensemble des données

Application de `partition()` sur tous les acteurs visibles pour identifier les clusters de doublons.

In [ ]:
import random
for key in random.sample(list(data_dict_visible.keys()), 3):
    print(key)
    print(data_dict_visible[key])

# To run deduplication by departement
# departement = "93"
# data_dict_visible_sample = {k: v for k, v in data_dict_visible.items() if v["code_postal"] and v["code_postal"].startswith(departement)}

# To run deduplication on all entries
data_dict_visible_sample = data_dict_visible

for key in random.sample(list(data_dict_visible_sample.keys()), 3):
    print(key)
    print(data_dict_visible_sample[key])


In [ ]:
%%time

from collections import defaultdict


def partition_with_postal_code_guardrail(deduper, data, threshold):
    """Garde-fou métier: impossible de rapprocher deux acteurs de code_postal différent.

    On exécute `deduper.partition` séparément par code_postal.
    Les acteurs sans code_postal restent en singletons.
    """

    groups = defaultdict(dict)
    singletons = []

    for record_id, record in data.items():
        cp = record.get("code_postal")
        if not cp:
            singletons.append(([record_id], [1.0]))
            continue
        groups[str(cp)][record_id] = record

    clustered = []
    for cp, group in groups.items():
        if len(group) == 1:
            only_id = next(iter(group))
            clustered.append(([only_id], [1.0]))
            continue
        clustered.extend(deduper.partition(group, threshold=threshold))

    clustered.extend(singletons)
    return clustered


# Lancer la déduplication uniquement sur les acteurs VISIBLES
print(f"Déduplication en cours avec seuil={best_threshold:.2f}...")
print(f"Nombre d'enregistrements (visibles uniquement) : {len(data_dict_visible)}")

print(f"Nombre d'enregistrements dans l'échantillon (visibles uniquement) : {len(data_dict_visible_sample)}")

clustered = partition_with_postal_code_guardrail(
    deduper,
    data_dict_visible_sample,
    threshold=best_threshold,
)

# Filtrer les clusters avec plus d'un élément (vrais doublons)
duplicate_clusters = [c for c in clustered if len(c[0]) > 1]

print(f"\nRésultats :")
print(f"  Clusters totaux : {len(clustered)}")
print(f"  Clusters de doublons (>1 élément) : {len(duplicate_clusters)}")
print(f"  Acteurs dans des clusters de doublons : "
      f"{sum(len(c[0]) for c in duplicate_clusters)}")


## 10. Analyse des résultats

In [ ]:
# Distribution de la taille des clusters
cluster_sizes = [len(c[0]) for c in duplicate_clusters]
size_counts = pd.Series(cluster_sizes).value_counts().sort_index()

print("Distribution de la taille des clusters de doublons :")
for size, count in size_counts.items():
    print(f"  Taille {size:3d} : {count:5d} clusters")

print(f"\n  Taille max : {max(cluster_sizes) if cluster_sizes else 0}")
print(f"  Taille moyenne : {np.mean(cluster_sizes):.1f}" if cluster_sizes else "")

In [ ]:
# Afficher les 20 premiers clusters de doublons trouvés
print("=== Exemples de clusters de doublons ===\n")

# Trier par score de confiance moyen décroissant
sorted_clusters = sorted(
    duplicate_clusters,
    key=lambda c: np.mean(c[1]),
    reverse=True,
)

for i, (cluster_ids, cluster_scores) in enumerate(sorted_clusters[:20]):
    avg_score = np.mean(cluster_scores)
    print(f"--- Cluster {i+1} (taille={len(cluster_ids)}, confiance={avg_score:.3f}) ---")

    for identifiant_unique, score in zip(cluster_ids, cluster_scores):
        record = data_dict_visible_sample.get(identifiant_unique, {})
        source = df.loc[identifiant_unique, "source__libelle"] if identifiant_unique in df.index else "?"
        print(f"  [{score:.3f}] {identifiant_unique}")
        print(f"           nom={record.get('nom', '?')}")
        print(f"           adresse={record.get('adresse', '?')}, "
              f"cp={record.get('code_postal', '?')}, ville={record.get('ville', '?')}")
        print(f"           siret={record.get('siret', '?')}, "
              f"tel={record.get('telephone', '?')}, source={source}")
    print()

In [ ]:
# Analyse croisée par source : quelles sources se recoupent ?
source_pairs = defaultdict(int)

for cluster_ids, _ in duplicate_clusters:
    sources_in_cluster = set()
    for identifiant_unique in cluster_ids:
        if identifiant_unique in df.index:
            source = df.loc[identifiant_unique, "source__libelle"]
            if source is not None:
                sources_in_cluster.add(source)
    for s1, s2 in combinations(sorted(sources_in_cluster), 2):
        source_pairs[(s1, s2)] += 1
    # Intra-source
    for s in sources_in_cluster:
        count_in_cluster = sum(
            1 for identifiant_unique in cluster_ids
            if identifiant_unique in df.index and df.loc[identifiant_unique, "source__libelle"] == s
        )
        if count_in_cluster > 1:
            source_pairs[(s, s)] += 1

print("=== Doublons inter/intra-sources (top 30) ===\n")
sorted_pairs = sorted(source_pairs.items(), key=lambda x: x[1], reverse=True)
for (s1, s2), count in sorted_pairs[:30]:
    label = f"{s1} x {s2}" if s1 != s2 else f"{s1} (intra)"
    print(f"  {count:5d}  {label}")

## 11. Comparaison avec les clusters existants

Vérification : les clusters existants (parent/enfants) sont-ils retrouvés par le modèle ?

In [ ]:
# Construire un mapping acteur -> cluster_id depuis les résultats dedupe
acteur_to_new_cluster = {}
for cluster_idx, (cluster_ids, _) in enumerate(clustered):
    for identifiant_unique in cluster_ids:
        acteur_to_new_cluster[identifiant_unique] = cluster_idx

# Construire un mapping acteur -> cluster existant depuis les parents

# get only parents with code_postal starting with departement code
parents = VueActeur.objects.filter(est_parent=True,
                                   code_postal__startswith=departement)

acteur_to_existing_cluster = {}
for parent in parents:
    enfants = parent.duplicats.all()
    cluster_members = [parent.identifiant_unique] + [
        e.identifiant_unique for e in enfants
    ]
    for identifiant_unique in cluster_members:
        acteur_to_existing_cluster[identifiant_unique] = parent.identifiant_unique

# Comparer
existing_clusters_by_parent = defaultdict(set)
for identifiant_unique, parent_id in acteur_to_existing_cluster.items():
    existing_clusters_by_parent[parent_id].add(identifiant_unique)

found = 0
partial = 0
missed = 0
total_existing = len(existing_clusters_by_parent)

for parent_id, existing_members in existing_clusters_by_parent.items():
    # Vérifier les membres présents dans le DF
    members_in_df = existing_members & set(df.index)
    if len(members_in_df) < 2:
        continue

    # Trouver les clusters dedupe correspondants
    new_clusters = set()
    for identifiant_unique in members_in_df:
        if identifiant_unique in acteur_to_new_cluster:
            new_clusters.add(acteur_to_new_cluster[identifiant_unique])

    if len(new_clusters) == 1:
        # Tous dans le même cluster dedupe
        found += 1
    elif len(new_clusters) > 1:
        # Répartis dans plusieurs clusters (partiellement trouvé)
        partial += 1
    else:
        missed += 1

print(f"=== Retrouvabilité des clusters existants ===")
print(f"  Total clusters existants : {total_existing}")
print(f"  Exactement retrouvés    : {found}")
print(f"  Partiellement retrouvés : {partial}")
print(f"  Non retrouvés           : {missed}")
print(f"  Taux de rappel exact    : {found/(found+partial+missed)*100:.1f}%"
      if (found+partial+missed) > 0 else "")

## 12. Nouveaux doublons détectés (non encore clusterisés)

Clusters trouvés par dedupe qui ne correspondent à aucun cluster existant -- ce sont les **nouveaux doublons potentiels**.

In [ ]:
# Identifier les clusters qui contiennent uniquement des acteurs non-déjà-clusterisés
new_clusters = []

for cluster_ids, cluster_scores in duplicate_clusters:
    # Vérifier si aucun des membres n'est déjà dans un cluster existant
    already_clustered = any(
        identifiant_unique in acteur_to_existing_cluster for identifiant_unique in cluster_ids
    )
    if not already_clustered:
        new_clusters.append((cluster_ids, cluster_scores))

print(f"Nouveaux clusters de doublons potentiels : {len(new_clusters)}")
print(f"Acteurs concernés : {sum(len(c[0]) for c in new_clusters)}")

# Afficher les 30 premiers nouveaux clusters (triés par confiance)
new_sorted = sorted(new_clusters, key=lambda c: np.mean(c[1]), reverse=True)

print(f"\n=== Top 30 nouveaux doublons (par confiance) ===\n")
for i, (cluster_ids, cluster_scores) in enumerate(new_sorted[:30]):
    avg_score = np.mean(cluster_scores)
    print(f"--- Nouveau cluster {i+1} (taille={len(cluster_ids)}, "
          f"confiance={avg_score:.3f}) ---")
    for identifiant_unique, score in zip(cluster_ids, cluster_scores):
        record = data_dict_all.get(identifiant_unique, {})
        source = df.loc[identifiant_unique, "source__libelle"] if identifiant_unique in df.index else "?"
        print(f"  [{score:.3f}] {record.get('nom', '?')}")
        print(f"           {record.get('adresse', '?')}, "
              f"{record.get('code_postal', '?')} {record.get('ville', '?')}")
        print(f"           siret={record.get('siret', '?')} | source={source}")
    print()

## 13. Export des résultats

Export des clusters de doublons en CSV pour analyse ultérieure.

In [ ]:
# Construire un DataFrame avec tous les clusters de doublons
from django.contrib.admin.utils import quote as dj_quote
from urllib.parse import quote

rows = []
for cluster_idx, (cluster_ids, cluster_scores) in enumerate(duplicate_clusters):
    for identifiant_unique, score in zip(cluster_ids, cluster_scores):
        record = data_dict_all.get(identifiant_unique, {})
        is_new = identifiant_unique not in acteur_to_existing_cluster
        location = record.get("location")
        latitude = location[1] if location else None
        longitude = location[0] if location else None
        rows.append({
            "cluster_id": cluster_idx,
            "confiance": score,
            "identifiant_unique": identifiant_unique,
            "uuid": record.get("uuid"),
            "url": f"https://quefairedemesdechets.ademe.fr/admin/qfdmo/vueacteur/{dj_quote(identifiant_unique).replace(' ', '%20')}/change/",
            "latitude": latitude,
            "longitude": longitude,
            "nom": record.get("nom"),
            "acteur_type": record.get("acteur_type__libelle"),
            "adresse": record.get("adresse"),
            "adresse_complement": record.get("adresse_complement"),
            "code_postal": record.get("code_postal"),
            "ville": record.get("ville"),
            "siret": record.get("siret"),
            "telephone": record.get("telephone"),
            "email": record.get("email"),
            "source": df.loc[identifiant_unique, "source__libelle"] if identifiant_unique in df.index else None,
            "nouveau_doublon": is_new,
            "parent_existant": acteur_to_existing_cluster.get(identifiant_unique),
        })

df_clusters = pd.DataFrame(rows)

# Sauvegarder
output_path = Path("deduplication_clusters.csv")
df_clusters.to_csv(output_path, index=False)

print(f"Export : {output_path}")
print(f"  {len(df_clusters)} lignes, {df_clusters['cluster_id'].nunique()} clusters")
print(f"  Nouveaux doublons : {df_clusters['nouveau_doublon'].sum()} acteurs")
print(f"\nMétadonnées du modèle :")
print(f"  Seuil : {best_threshold:.2f}")
print(f"  F1-score (test) : {best_f1:.3f}")
print(f"  Paires match : {len(match_set)}")
print(f"  Paires distinct : {len(distinct_set)}")

df_clusters.head(10)